In [2]:
# ================== PROCESO COMPLETO: Configuración, Diagnóstico, Análisis y Exportación ==================
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from shapely.geometry import Point
from folium.plugins import MarkerCluster
from datetime import datetime
from geopy.distance import geodesic
import openpyxl
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Font, PatternFill, Alignment
from io import BytesIO
from IPython.display import display, IFrame
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')

# --- Configuración de parámetros ---
input_folder = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input'
archivos = [f for f in os.listdir(input_folder) if f.lower().endswith(('.xlsx', '.xls'))]
if archivos:
    ARCHIVO_DATOS = os.path.join(input_folder, archivos[0])
    print(f'Archivo de entrada detectado: {ARCHIVO_DATOS}')
else:
    raise FileNotFoundError('No se encontró ningún archivo Excel en la carpeta Input')
DIRECTORIO_RESULTADOS = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados'
CANTIDAD_CLIENTES_ANALIZAR = None  # Procesar todos los clientes
TOLERANCIA_NEUTRAL_KM = 2
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(DIRECTORIO_RESULTADOS, exist_ok=True)

# --- Diagnóstico de coordenadas en Clientes ---
try:
    df_clientes_diag = pd.read_excel(ARCHIVO_DATOS, sheet_name='Clientes')
    lat_col = 'U_latitud'
    lon_col = 'U_longitud'
    nulos_lat = df_clientes_diag[lat_col].isnull().sum()
    nulos_lon = df_clientes_diag[lon_col].isnull().sum()
    no_num_lat = pd.to_numeric(df_clientes_diag[lat_col], errors='coerce').isnull() & df_clientes_diag[lat_col].notnull()
    no_num_lon = pd.to_numeric(df_clientes_diag[lon_col], errors='coerce').isnull() & df_clientes_diag[lon_col].notnull()
    print(f"Valores nulos en {lat_col}: {nulos_lat}")
    print(f"Valores nulos en {lon_col}: {nulos_lon}")
    print(f"Valores no numéricos en {lat_col}: {no_num_lat.sum()}")
    print(f"Valores no numéricos en {lon_col}: {no_num_lon.sum()}")
    if nulos_lat > 0 or nulos_lon > 0 or no_num_lat.sum() > 0 or no_num_lon.sum() > 0:
        print('Filas problemáticas:')
        display(df_clientes_diag[no_num_lat | no_num_lon | df_clientes_diag[lat_col].isnull() | df_clientes_diag[lon_col].isnull()])
        raise Exception('Corrige los valores nulos o no numéricos en las coordenadas antes de continuar.')
    else:
        print('No se detectaron problemas de nulos o no numéricos en las coordenadas de Clientes.')
except Exception as e:
    print(f'Error en el diagnóstico de coordenadas: {e}')
    raise

# --- Carga de datos ---
def cargar_datos(archivo):
    try:
        df_clientes = pd.read_excel(archivo, sheet_name='Clientes')
        df_cdr = pd.read_excel(archivo, sheet_name='CDR')
        lat_col = 'U_latitud'
        lon_col = 'U_longitud'
        if lat_col not in df_clientes.columns or lon_col not in df_clientes.columns:
            raise ValueError(f"No se encontraron columnas '{lat_col}' y '{lon_col}' en 'Clientes'.")
        if lat_col not in df_cdr.columns or lon_col not in df_cdr.columns:
            raise ValueError(f"No se encontraron columnas '{lat_col}' y '{lon_col}' en 'CDR'.")
        gdf_clientes = gpd.GeoDataFrame(df_clientes, geometry=gpd.points_from_xy(df_clientes[lon_col], df_clientes[lat_col]))
        gdf_cdr = gpd.GeoDataFrame(df_cdr, geometry=gpd.points_from_xy(df_cdr[lon_col], df_cdr[lat_col]))
        return gdf_clientes, gdf_cdr
    except Exception as e:
        print(f'Error al cargar datos: {e}')
        return None, None

gdf_clientes, gdf_cdr = cargar_datos(ARCHIVO_DATOS)
if gdf_clientes is None or gdf_cdr is None:
    raise Exception('No se pudieron cargar los datos')
if gdf_cdr.empty:
    raise ValueError('La hoja CDR está vacía. Verifica el archivo de entrada.')
print(f'CDRs detectados: {gdf_cdr["CodMET"].unique().tolist()}')
print(f'Clientes cargados: {len(gdf_clientes)}')
print(f'Columnas en Clientes: {gdf_clientes.columns.tolist()}')
print(f'Columnas en CDR: {gdf_cdr.columns.tolist()}')
columnas_requeridas = ['U_latitud','U_longitud']
for col in columnas_requeridas:
    if col not in gdf_clientes.columns:
        raise ValueError(f'Falta la columna {col} en la hoja Clientes')
    if col not in gdf_cdr.columns:
        raise ValueError(f'Falta la columna {col} en la hoja CDR')

# --- Análisis de distancias dinámico ---
def distancia_geodesica(coord1, coord2):
    return geodesic(coord1, coord2).km

def analizar_distancias(gdf_cli, gdf_cdr, cantidad=None, tolerancia=2):
    if cantidad and cantidad < len(gdf_cli):
        gdf_cli = gdf_cli.sample(n=cantidad, random_state=42)
    for df, nombre in [(gdf_cli, 'Clientes'), (gdf_cdr, 'CDR')]:
        for col in ['U_latitud', 'U_longitud']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].isnull().any():
                raise ValueError(f'Valores no numéricos o nulos en la columna {col} de la hoja {nombre}')
    cdrs = gdf_cdr[['CodMET', 'U_latitud', 'U_longitud']].to_dict('records')
    resultados = []
    for _, cli in gdf_cli.iterrows():
        distancias = []
        for cdr in cdrs:
            dist = distancia_geodesica((cli['U_latitud'], cli['U_longitud']), (cdr['U_latitud'], cdr['U_longitud']))
            distancias.append({'CodMET': cdr['CodMET'], 'Dist_km': dist})
        distancias = sorted(distancias, key=lambda x: x['Dist_km'])
        mejor_cdr = distancias[0]['CodMET']
        mejor_dist = distancias[0]['Dist_km']
        if len(distancias) > 1:
            segunda_dist = distancias[1]['Dist_km']
            diferencia = abs(mejor_dist - segunda_dist)
        else:
            segunda_dist = None
            diferencia = None
        if diferencia is not None and diferencia <= tolerancia:
            mejor_opcion = 'Neutral'
        else:
            mejor_opcion = mejor_cdr
        resultado = {
            'CodCli': cli.get('CodCli', ''),
            'Cliente': cli.get('Codigo_Cliente', ''),
            'Nombre': cli.get('Nombre', ''),
            'ID': cli.get('ID', ''),
            'Ruta': cli.get('Ruta', ''),
            'U_latitud': cli['U_latitud'],
            'U_longitud': cli['U_longitud'],
            'Mejor_Opcion': mejor_opcion,
            'CDR_Mas_Cercano': mejor_cdr,
            'Dist_Mas_Cercano_km': round(mejor_dist,2),
            'Dist_Segundo_km': round(segunda_dist,2) if segunda_dist is not None else None,
            'Diferencia_km': round(diferencia,2) if diferencia is not None else None,
        }
        for d in distancias:
            resultado[f'Dist_{d["CodMET"]}_km'] = round(d['Dist_km'],2)
        resultados.append(resultado)
    return pd.DataFrame(resultados)

df_analisis = analizar_distancias(gdf_clientes, gdf_cdr, cantidad=CANTIDAD_CLIENTES_ANALIZAR, tolerancia=TOLERANCIA_NEUTRAL_KM)
print('Primeras filas del análisis:')
display(df_analisis.head())

# --- Visualización y exportación ---
def crear_visualizaciones(df_analisis):
    buf = BytesIO()
    asignacion = df_analisis['Mejor_Opcion'].value_counts()
    colores = sns.color_palette('tab10', len(asignacion)).as_hex()
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    axs[0].pie(asignacion.values, labels=asignacion.index, autopct='%1.1f%%', colors=colores, startangle=90)
    axs[0].set_title('Distribución de Asignación de Clientes', fontsize=13, color='#FF6B00')
    axs[1].hist(df_analisis['Dist_Mas_Cercano_km'], bins=25, alpha=0.7, color='#00529B', edgecolor='black')
    axs[1].set_xlabel('Distancia al CDR más cercano (km)', fontsize=11)
    axs[1].set_ylabel('Frecuencia', fontsize=11)
    axs[1].set_title('Histograma de Distancias al CDR más cercano', fontsize=13, color='#FF6B00')
    axs[1].grid(True, alpha=0.3)
    cdrs = [c for c in asignacion.index if c != 'Neutral']
    colores_disp = dict(zip(cdrs, colores))
    for cdr in cdrs:
        subset = df_analisis[df_analisis['Mejor_Opcion'] == cdr]
        axs[2].scatter(subset['U_longitud'], subset['U_latitud'], s=20, alpha=0.7, c=colores_disp[cdr], label=cdr)
    if 'Neutral' in asignacion.index:
        subset = df_analisis[df_analisis['Mejor_Opcion'] == 'Neutral']
        axs[2].scatter(subset['U_longitud'], subset['U_latitud'], s=20, alpha=0.7, c='#FFA500', label='Neutral')
    axs[2].set_xlabel('Longitud', fontsize=11)
    axs[2].set_ylabel('Latitud', fontsize=11)
    axs[2].set_title('Dispersión de Clientes por CDR Asignado', fontsize=13, color='#FF6B00')
    axs[2].legend(fontsize=11)
    plt.tight_layout()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

def crear_mapa_cluster(df_analisis, gdf_cdr, directorio, timestamp):
    centro_lat = df_analisis['U_latitud'].mean()
    centro_lon = df_analisis['U_longitud'].mean()
    mapa = folium.Map(location=[centro_lat, centro_lon], zoom_start=8, tiles='OpenStreetMap')
    rutas = df_analisis['Ruta'].unique() if 'Ruta' in df_analisis.columns else ['Todos']
    colores_rutas = sns.color_palette('husl', len(rutas)).as_hex()
    for idx, ruta in enumerate(rutas):
        grupo = df_analisis[df_analisis['Ruta'] == ruta] if 'Ruta' in df_analisis.columns else df_analisis
        capa_ruta = folium.FeatureGroup(name=f'Ruta: {ruta}')
        cluster = MarkerCluster(name=f'Clientes {ruta}')
        if len(grupo) > 2:
            from shapely.geometry import MultiPoint
            puntos = [Point(row['U_longitud'], row['U_latitud']) for _, row in grupo.iterrows()]
            multipoint = MultiPoint(puntos)
            hull = multipoint.convex_hull
            if hull.geom_type == 'Polygon':
                coords = [(lat, lon) for lon, lat in hull.exterior.coords]
                popup_ruta = f'''<div style="width:210px; padding:9px 10px; background:{colores_rutas[idx]}; color:#fff; border-radius:7px; box-shadow:0 1px 6px #aaa; font-size:9px; font-weight:bold; text-align:center;"><span style='font-size:11px;'>🛣️ Ruta: {ruta}</span></div>'''
                folium.Polygon(locations=coords, color=colores_rutas[idx], fill=True, fill_opacity=0.25, popup=folium.Popup(popup_ruta, max_width=225)).add_to(capa_ruta)
        for _, row in grupo.iterrows():
            popup_html = f'''<div style="width:420px; padding:16px 18px; background:#fff; border-radius:14px; box-shadow:0 2px 12px #aaa; font-size:17px;"><div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;"><span style="font-weight:bold; color:{colores_rutas[idx]}; font-size:20px;">Cliente: {row.get('CodCli','')}</span><span style="font-size:28px;">🧑‍💼</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>📚 <b>Nombre:</b> {row.get('Nombre','')}</span><span>🛣️ <b>Ruta:</b> {row.get('Ruta','')}</span></div><div style="display:flex; justify-content:space-between; margin-bottom:8px;"><span>🎯 <b>Mejor:</b> {row.get('Mejor_Opcion','')}</span><span>💡 <b>Diferencia:</b> {row.get('Diferencia_km','')} km</span></div></div>'''
            folium.Marker(location=[row['U_latitud'], row['U_longitud']], popup=folium.Popup(popup_html, max_width=450), icon=folium.Icon(color='blue', icon='user', prefix='fa')).add_to(cluster)
        cluster.add_to(capa_ruta)
        capa_ruta.add_to(mapa)
    colores_cdr = sns.color_palette('tab10', len(gdf_cdr)).as_hex()
    for idx, (_, cdr) in enumerate(gdf_cdr.iterrows()):
        meet_color = colores_cdr[idx % len(colores_cdr)]
        meet_icon = '🏢'
        popup_html = f'''<div style="width:320px; padding:14px; background:{meet_color}; color:#fff; border-radius:12px; box-shadow:0 2px 8px #aaa; text-align:center; font-size:18px;"><div style="font-size:36px;">{meet_icon}</div><div style="font-weight:bold; font-size:22px; margin-top:6px;">{cdr['CodMET']}</div><div style="font-size:16px; margin-top:4px;">Lat: {cdr['U_latitud']}<br>Lon: {cdr['U_longitud']}</div></div>'''
        folium.Marker(location=[cdr['U_latitud'], cdr['U_longitud']], popup=folium.Popup(popup_html, max_width=350), icon=folium.Icon(color='red', icon='building', prefix='fa')).add_to(mapa)
    folium.LayerControl(collapsed=False, position='topleft').add_to(mapa)
    archivo_mapa = os.path.join(directorio, f'Mapa_Cluster_{timestamp}.html')
    mapa.save(archivo_mapa)
    return archivo_mapa

def exportar_excel(df_analisis, directorio, timestamp, cantidad, grafica_buf):
    archivo_excel = os.path.join(directorio, f'Analisis_CDR_Adaptable_{len(df_analisis)}_Clientes_{timestamp}.xlsx')
    with pd.ExcelWriter(archivo_excel, engine='openpyxl') as writer:
        resumen = [['📊 MÉTRICAS PRINCIPALES', '']]
        resumen.append(['👥 Total clientes analizados', len(df_analisis)])
        for cdr in df_analisis['Mejor_Opcion'].unique():
            if cdr == 'Neutral':
                resumen.append(['⚖️ Neutrales', len(df_analisis[df_analisis['Mejor_Opcion']=='Neutral'])])
            else:
                resumen.append([f'🏢 {cdr}', len(df_analisis[df_analisis['Mejor_Opcion']==cdr])])
        resumen.append(['💰 Distancia total (km)', round(df_analisis['Dist_Mas_Cercano_km'].sum(),2)])
        resumen.append(['📈 Distancia promedio (km)', round(df_analisis['Dist_Mas_Cercano_km'].mean(),2)])
        resumen.append(['🏆 Mínima distancia individual (km)', round(df_analisis['Dist_Mas_Cercano_km'].min(),2)])
        resumen.append(['🔢 Mediana distancia (km)', round(df_analisis['Dist_Mas_Cercano_km'].median(),2)])
        for cdr in df_analisis['Mejor_Opcion'].unique():
            porcentaje = round(len(df_analisis[df_analisis['Mejor_Opcion']==cdr])/len(df_analisis)*100,1)
            resumen.append([f'📊 % {cdr}', porcentaje])
        resumen.append(['🗓️ Fecha análisis', timestamp])
        resumen.append(['📄 Archivo fuente', os.path.basename(directorio)])
        df_resumen = pd.DataFrame(resumen, columns=['Métrica','Valor'])
        df_resumen.to_excel(writer, sheet_name='Resumen_Ejecutivo', index=False)
        df_analisis.to_excel(writer, sheet_name='Analisis_Detallado', index=False)
        if 'Ruta' in df_analisis.columns:
            resumen_ruta = df_analisis.groupby('Ruta').agg({
                'Cliente':'count',
                'Dist_Mas_Cercano_km':'sum',
                'Mejor_Opcion':lambda x: x.value_counts().to_dict()
            }).rename(columns={'Cliente':'Total_Clientes','Dist_Mas_Cercano_km':'Distancia_Total'})
            resumen_ruta.to_excel(writer, sheet_name='Resumen_por_Ruta')
    wb = openpyxl.load_workbook(archivo_excel)
    ws = wb['Resumen_Ejecutivo']
    img = XLImage(grafica_buf)
    img.width = 420
    img.height = 220
    img.anchor = 'C2'
    ws.add_image(img)
    naranja_oscuro = 'C25A00'
    for row in ws.iter_rows(min_row=2, max_row=2, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(bold=True, color='FFFFFF', size=14)
            cell.fill = PatternFill(start_color=naranja_oscuro, end_color=naranja_oscuro, fill_type='solid')
            cell.alignment = Alignment(horizontal='center')
    for row in ws.iter_rows(min_row=3, max_row=ws.max_row, min_col=1, max_col=2):
        for cell in row:
            cell.font = Font(size=12)
            cell.alignment = Alignment(horizontal='left')
            if row[0].value and 'Neutral' in str(row[0].value):
                cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            elif row[0].value and 'Distancia' in str(row[0].value):
                cell.fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')
    ws.column_dimensions['A'].width = 32
    ws.column_dimensions['B'].width = 22
    wb.save(archivo_excel)
    return archivo_excel

# --- Generar visualizaciones, mapa y exportar ---
grafica_buf = crear_visualizaciones(df_analisis)
archivo_mapa = crear_mapa_cluster(df_analisis, gdf_cdr, DIRECTORIO_RESULTADOS, timestamp)
archivo_excel = exportar_excel(df_analisis, DIRECTORIO_RESULTADOS, timestamp, CANTIDAD_CLIENTES_ANALIZAR, grafica_buf)
print(f'Mapa generado en: {archivo_mapa}')
print(f'Reporte Excel generado en: {archivo_excel}')
display(IFrame(src=archivo_mapa, width=900, height=600))

Archivo de entrada detectado: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input\Clientes_Ruta2.xlsx
Valores nulos en U_latitud: 0
Valores nulos en U_longitud: 0
Valores no numéricos en U_latitud: 0
Valores no numéricos en U_longitud: 0
No se detectaron problemas de nulos o no numéricos en las coordenadas de Clientes.
Valores nulos en U_latitud: 0
Valores nulos en U_longitud: 0
Valores no numéricos en U_latitud: 0
Valores no numéricos en U_longitud: 0
No se detectaron problemas de nulos o no numéricos en las coordenadas de Clientes.
CDRs detectados: ['Ceylan', 'CDMX2']
Clientes cargados: 11631
Columnas en Clientes: ['U_latitud', 'U_longitud', 'CodCli', 'Ruta', 'Nombre', 'ID', 'geometry']
Columnas en CDR: ['U_latitud', 'U_longitud', 'CodMET', 'geometry']
CDRs detectados: ['Ceylan', 'CDMX2']
Clientes cargados: 11631
Columnas en Clientes: ['U_latitud', 'U_longitud', 'CodCli', 'Ruta', 'Nombre', 'ID', 'geometry']
Columnas en CDR: ['U_latitud', 'U_longitud', '

,CodCli,Cliente,Nombre,ID,Ruta,U_latitud,U_longitud,Mejor_Opcion,CDR_Mas_Cercano,Dist_Mas_Cercano_km,Dist_Segundo_km,Diferencia_km,Dist_Ceylan_km,Dist_CDMX2_km
0,5003746,,50037461,1,1,19.61048,-98.97167,Ceylan,Ceylan,24.24,27.56,3.32,24.24,27.56
1,5003748,,50037481,2,1,19.64471,-98.88292,Neutral,CDMX2,33.17,34.26,1.09,34.26,33.17
2,5003753,,50037531,3,1,19.58583,-98.95072,Neutral,CDMX2,25.08,25.16,0.08,25.16,25.08
3,5003755,,50037551,4,1,19.58842,-98.95764,Neutral,Ceylan,24.58,25.26,0.69,24.58,25.26
4,5003757,,50037571,5,1,19.67191,-98.93044,Ceylan,Ceylan,31.48,34.83,3.35,31.48,34.83


Mapa generado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados\Mapa_Cluster_20250922_154457.html
Reporte Excel generado en: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Resultados\Analisis_CDR_Adaptable_11631_Clientes_20250922_154457.xlsx
